In [2]:
!module load python

In [3]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

is_cuda = torch.cuda.is_available()
num = torch.cuda.device_count()
name = torch.cuda.get_device_name(0) if is_cuda and num > 0 else "None"

print(f"CUDA available: {is_cuda}")
print(f"GPU count: {num}")
print(f"Primary GPU: {name}")

CUDA available: True
GPU count: 1
Primary GPU: NVIDIA H100 80GB HBM3 MIG 3g.40gb


In [4]:
device = "cuda"

In [5]:
hub = Path(os.path.expanduser("~/.cache/huggingface/hub"))
base = hub / "models--codellama--CodeLlama-7b-Instruct-hf"

# Prefer the ref in refs/main; fall back to the newest snapshot
ref_file = base / "refs" / "main"
commit = ref_file.read_text().strip()

MODEL_PATH = str(base / "snapshots" / commit)
print("MODEL_PATH =", MODEL_PATH)

MODEL_PATH = /home/rpinter/.cache/huggingface/hub/models--codellama--CodeLlama-7b-Instruct-hf/snapshots/22cb240e0292b0b5ab4c17ccd97aa3a2f799cbed


In [6]:
tok = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True,
    local_files_only=True,
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from transformers import TextStreamer
import torch

def ask(prompt, max_new_tokens=128, temperature=0.2, top_p=0.9):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.eos_token_id,
        )

    # Return only the newly generated part (skip the prompt tokens)
    return tok.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Simple smoke tests
print("\n--- Test 1 ---")
print(ask("Say hello in one short sentence."))

print("\n--- Test 2 (tiny code task) ---")
print(ask("Write a Python function reverse_str(s) that returns the reversed string, then show a quick example."))


--- Test 1 ---
"Hello!"

--- Test 2 (tiny code task) ---
Sure! Here's a Python function that reverses a string:
```
def reverse_str(s):
    return s[::-1]
```
And here's a quick example of how to use it:
```
>>> reverse_str("hello")
"lohel"
```
Note that the `[::-1]` syntax is used to reverse the string. The `[::-1]` syntax is called a "slice" and it selects a range of elements from a sequence (such as a string) in reverse order. In this case, it selects all the elements of


In [8]:
import torch

def ask_fast(prompt, max_new_tokens=32):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [9]:
print(ask("""
Given this entities:
- particle
- geometric_shape
- line
- sound_sample
- rhythm
- sound_envelop
- oscillator
- video
- image
- text
- other

Read this code:

```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```

Now select the entities that are present in this code
"""))


The entities present in this code are:

* `particle`
* `geometric_shape`
* `line`
* `sound_sample`
* `rhythm`
* `sound_envelop`
* `oscillator`
* `video`
* `image`
* `text`
* `other`

The code defines a function `hal` that generates a grid of triangles using the `trianglewlines` function. The `trianglewlines` function takes four points as input and draws a triangle between them. The `hal` function


In [1]:
--

SyntaxError: invalid syntax (3659366440.py, line 1)